# Stage S: make the upload pack

Use a **Python 3.12 CPU runtime**. Run this only after the preregistration PR has been
reviewed and merged into main. This notebook does not contact Suno or download song-model weights.
It creates 12 original melody MIDI files, piano and Voice Oohs WAV files, lyric text,
and a recording table. Do not upload any owner-held-out melody.

Rendered outputs stay in the ignored E7 outputs folder. Model caches stay outside the clone.
Dependencies are installed in this disposable
runtime. The existing repository source is never edited. Keep the upload hash JSON and give
it to the reviewer before generating on Suno. Read `suno_protocol.md` for the three routes.

In [ ]:
import os, sys, subprocess, json
from pathlib import Path
sys.dont_write_bytecode = True
os.environ["PYTHONDONTWRITEBYTECODE"] = "1"
REPO = Path("/content/dacapo-e7")
WORK = Path("/content/e7-runtime")
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", "main",
                    "https://github.com/ninondev/DaCapo", str(REPO)], check=True)
else:
    dirty = subprocess.check_output(["git", "-C", str(REPO), "status", "--porcelain"], text=True)
    if dirty:
        raise RuntimeError("The runtime clone has local edits. Use a fresh Colab runtime.")
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "switch", "main"], check=True)
    subprocess.run(["git", "-C", str(REPO), "merge", "--ff-only", "origin/main"], check=True)
os.chdir(REPO)
sys.path.insert(0, str(REPO))
os.environ["PYTHONPATH"] = str(REPO)
if not (REPO / "experiments/e7_melody_adapter/prereg_suno.json").is_file():
    raise RuntimeError("Wait for the preregistration PR to be reviewed and merged into main.")
print("DaCapo commit:", subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip())
from experiments.e7_melody_adapter.colab_setup import checkout_state, verify_registered_files, prepare
checkout_state()
verify_registered_files()

In [ ]:
runtime = prepare("pack", WORK)
os.environ.update(runtime["environment"])
PACK_OUTPUT = REPO / "experiments/e7_melody_adapter/outputs/upload-pack"
subprocess.run(["bash", "experiments/e7_melody_adapter/run_stage.sh", "pack",
                "--output", str(PACK_OUTPUT)], check=True, env=dict(os.environ))

In [ ]:
from google.colab import files
hashes = json.loads((PACK_OUTPUT / "suno_upload_hashes.json").read_text())
assert hashes["environment"]["git_commit"] != "unknown"
assert hashes["environment"]["git_dirty"] is False
checkout_state()
files.download(str(PACK_OUTPUT / "e7_suno_upload.zip"))
files.download(str(PACK_OUTPUT / "suno_upload_hashes.json"))

Keep both files. Give `suno_upload_hashes.json` to the reviewer. Follow the protocol after
the preregistration is merged. Retain both outputs of each single Suno request and record
the actual menu option, displayed model version, date, and credits. Skip unavailable routes.
The GPU scoring notebook additionally requires the reviewed E5 v2 merge and registration.